In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow:", tf.__version__)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20

TRAIN_DIR = "../dataset/train/images"
VAL_DIR = "../dataset/validation/images"

In [ ]:
for folder in os.listdir(TRAIN_DIR):
    path = os.path.join(TRAIN_DIR, folder)

    if os.path.isdir(path):
        print(folder, ":", len(os.listdir(path)))

In [ ]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=25,

    zoom_range=0.25,

    width_shift_range=0.15,

    height_shift_range=0.15,

    shear_range=0.15,

    brightness_range=[0.8,1.2],

    horizontal_flip=True,

    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_data = train_datagen.flow_from_directory(

    TRAIN_DIR,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

In [ ]:
val_data = val_datagen.flow_from_directory(

    VAL_DIR,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

In [ ]:
print("Train Images:", train_data.samples)

print("Validation Images:", val_data.samples)

print(train_data.class_indices)

print(val_data.class_indices)

In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10,10))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    plt.axis("off")

plt.show()

In [ ]:
base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)
)

base_model.trainable = False

In [ ]:
model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    BatchNormalization(),

    Dense(
        256,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        train_data.num_classes,
        activation="softmax"
    )
])

In [ ]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.2,

    patience=2,

    verbose=1
)

In [ ]:
model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=EPOCHS,

    callbacks=[

        early_stop,

        reduce_lr
    ]
)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.show()

In [ ]:
print(
    "Best Training Accuracy:",
    max(history.history["accuracy"])
)

print(
    "Best Validation Accuracy:",
    max(history.history["val_accuracy"])
)

In [ ]:
model.save(
    "../backend/model/steel_defect_mobilenet.keras"
)

print("Model Saved Successfully!")

In [ ]:
from tensorflow.keras.models import load_model

test_model = load_model(
    "../backend/model/steel_defect_mobilenet.keras",
    compile=False
)

print("Model Loaded Successfully!")

In [ ]:
import os

print(os.listdir("../dataset/validation/images/scratches")[:20])

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "../dataset/validation/images/scratches/scratches_242.jpg"

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

prediction = test_model.predict(img_array)

class_names = list(train_data.class_indices.keys())

print("Prediction:",
      class_names[np.argmax(prediction)])

print("Confidence:",
      np.max(prediction) * 100)

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(
    "../backend/model/steel_defect_mobilenet.keras",
    compile=False
)

model.save("../backend/model/steel_defect_mobilenet.h5")

print("H5 model saved")

In [ ]:
model.save_weights(
    "../backend/model/steel_weights.weights.h5"
)